# 03a — Balancing Authority Territories

**Purpose:** Download EIA balancing authority (BA) service territory shapefiles,
spatial-join the power plant points from notebook 01 into their host BA,
and produce a BA-level summary of installed capacity by fuel type.

Balancing authorities are the functional 'organs' of grid metabolism — each one
independently balances real-time supply and demand within its territory.
Mapping generation into BAs is the prerequisite for the flow graph in 03b.

**Inputs:**
- `data/processed/power_plants.geojson` — from notebook 01
- EIA BA shapefile (downloaded in this notebook via direct URL)

**Outputs:**
- `data/processed/ba_territories.geojson` — BA polygons with capacity summary attributes
- `data/processed/power_plants_with_ba.geojson` — plant points with BA code attached
- `data/processed/ba_capacity_summary.csv` — BA × fuel type capacity table
- `data/processed/ba_map.html` — Folium choropleth of total capacity per BA

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import requests
import zipfile
import io
import pandas as pd
import geopandas as gpd
import folium
import utils

## 1. Download EIA BA Territory Shapefile

EIA publishes BA service territory boundaries as part of their annual
electric power survey geography. The file is available as a zip of shapefiles.

**Source:**
https://www.eia.gov/electricity/data/eia861/

The direct shapefile URL pattern is:
`https://www.eia.gov/electricity/data/eia861/zip/shapefiles.zip`

If that URL fails (EIA occasionally reorganises files), fall back to:
- Manual download from the EIA 861 page above
- Extract the `Balancing_Authority` folder into `data/raw/ba_shapefiles/`
- Re-run from the next cell

In [2]:
# ── Download and extract BA shapefiles ────────────────────────────────────────
BA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ba_shapefiles'
BA_RAW_DIR.mkdir(parents=True, exist_ok=True)

# EIA's original shapefile URL is no longer active; fall back to the HIFLD
# ArcGIS FeatureServer which has the same BA territories with EIA abbreviations.
ARCGIS_BA_URL = (
    'https://services5.arcgis.com/bsqU0jSPAuI04L89/arcgis/rest/services/'
    'Balancing_Authorities/FeatureServer/0/query'
    '?where=1%3D1&outFields=*&f=geojson&resultRecordCount=200'
)
GEOJSON_PATH = BA_RAW_DIR / 'Balancing_Authorities.geojson'

if GEOJSON_PATH.exists():
    print(f'BA GeoJSON already present: {GEOJSON_PATH}')
    print('Skipping download.')
else:
    print('Downloading BA territories from HIFLD ArcGIS FeatureServer...')
    r = requests.get(ARCGIS_BA_URL, timeout=60)
    r.raise_for_status()
    GEOJSON_PATH.write_bytes(r.content)
    print(f'Saved → {GEOJSON_PATH}  ({GEOJSON_PATH.stat().st_size / 1024:.0f} KB)')

BA GeoJSON already present: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/raw/ba_shapefiles/Balancing_Authorities.geojson
Skipping download.


In [3]:
# ── Locate the BA data file ────────────────────────────────────────────────────
# Accept either GeoJSON (downloaded above) or legacy shapefiles
geojson_files = list(BA_RAW_DIR.rglob('*.geojson'))
shp_files = [
    f for f in BA_RAW_DIR.rglob('*.shp')
    if any(kw in f.stem for kw in ['Balancing', 'balancing', 'BA', '_ba_'])
]

if geojson_files:
    ba_shp = geojson_files[0]
    print(f'Using GeoJSON: {ba_shp}')
elif shp_files:
    ba_shp = shp_files[0]
    print(f'Using shapefile: {ba_shp}')
else:
    print('WARNING: No BA data file found. Set ba_shp manually.')
    for f in BA_RAW_DIR.rglob('*'):
        print(f'  {f}')
    ba_shp = None

Using GeoJSON: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/raw/ba_shapefiles/Balancing_Authorities.geojson


## 2. Load and Inspect BA Territories

In [4]:
# ── Read BA shapefile ─────────────────────────────────────────────────────────
assert ba_shp is not None, 'Set ba_shp to the correct .shp path before continuing.'

ba = gpd.read_file(ba_shp)
print(f'Shape: {ba.shape}')
print(f'CRS:   {ba.crs}')
print(f'\nColumns: {ba.columns.tolist()}')
ba.head(3)

Shape: (85, 13)
CRS:   EPSG:4326

Columns: ['FID', 'BAL_AUTH', 'Area_sq_mi', 'BAL_AUTHID', 'Layer_ID', 'Rec_ID', 'NETGENMWH', 'NETGENRNG', 'Color_Code', 'BA_Abbrev', 'Shape__Area', 'Shape__Length', 'geometry']


,FID,BAL_AUTH,Area_sq_mi,BAL_AUTHID,Layer_ID,Rec_ID,NETGENMWH,NETGENRNG,Color_Code,BA_Abbrev,Shape__Area,Shape__Length,geometry
0,1,Alberta Electric System Operator,254817.421433,614256,84,2865,-99,Not Reported,3,AESO,93.577985,41.169971,"POLYGON ((-110 60, -110 59, -110 58, -110 57.0..."
1,2,Alcoa Power Generating Inc Yadkin Division,311.645951,604932,84,2918,1283511,"0 to 7,500,000",4,YAD,0.080234,1.154542,"POLYGON ((-80.37749 35.62192, -80.3703 35.6915..."
2,3,Anchorage AK (City of),1824.620395,423470,84,2866,-99,Not Reported,6,AMPL,0.792124,4.298141,"POLYGON ((-150.07545 61.15627, -150.07054 61.1..."


In [5]:
# ── Reproject to EPSG:4326 if needed ─────────────────────────────────────────
if ba.crs and ba.crs.to_epsg() != 4326:
    print(f'Reprojecting {ba.crs} → EPSG:4326...')
    ba = ba.to_crs(epsg=4326)
else:
    print('Already EPSG:4326.')

# ── Identify the BA code column ───────────────────────────────────────────────
# EIA shapefiles typically use 'EIACode', 'BA_CODE', or 'BACODE'.
# Inspect and confirm, then set BA_CODE_COL below.
print('\nCandidate BA code columns (look for short string codes like MISO, PJM, ERCO):')
for col in ba.columns:
    sample = ba[col].dropna().head(3).tolist()
    print(f'  {col}: {sample}')

Already EPSG:4326.

Candidate BA code columns (look for short string codes like MISO, PJM, ERCO):
  FID: [1, 2, 3]
  BAL_AUTH: ['Alberta Electric System Operator', 'Alcoa Power Generating Inc Yadkin Division', 'Anchorage AK (City of)']
  Area_sq_mi: [254817.4214332706, 311.645951172264, 1824.6203954820928]
  BAL_AUTHID: [614256, 604932, 423470]
  Layer_ID: [84, 84, 84]
  Rec_ID: [2865, 2918, 2866]
  NETGENMWH: [-99, 1283511, -99]
  NETGENRNG: ['Not Reported', '0 to 7,500,000', 'Not Reported']
  Color_Code: [3, 4, 6]
  BA_Abbrev: ['AESO', 'YAD', 'AMPL']
  Shape__Area: [93.57798496039732, 0.08023411318868057, 0.7921244434428445]
  Shape__Length: [41.16997125500643, 1.1545423264166002, 4.298140885494404]
  geometry: [<POLYGON ((-110 60, -110 59, -110 58, -110 57.025, -110 57, -110 56.375, -11...>, <POLYGON ((-80.377 35.622, -80.37 35.692, -80.352 35.718, -80.301 35.739, -8...>, <POLYGON ((-150.075 61.156, -150.071 61.163, -150.068 61.166, -150.046 61.18...>]


In [6]:
# ── Set BA code and name columns (edit if column names differ) ────────────────
# HIFLD FeatureServer uses BA_Abbrev (EIA code) and BAL_AUTH (full name).
# Legacy EIA shapefiles used EIACode / NAME — update these if using those files.
BA_CODE_COL = 'BA_Abbrev'   # EIA short code, e.g. 'MISO', 'PJM', 'ERCO'
BA_NAME_COL = 'BAL_AUTH'    # Full name

print(f'Unique BAs: {ba[BA_CODE_COL].nunique()}')
print(ba[[BA_CODE_COL, BA_NAME_COL]].drop_duplicates().head(10).to_string(index=False))

Unique BAs: 85
BA_Abbrev                                   BAL_AUTH
     AESO           Alberta Electric System Operator
      YAD Alcoa Power Generating Inc Yadkin Division
     AMPL                     Anchorage AK (City of)
     AZPS                  Arizona Public Service Co
     DEAA                       Arlington Valley LLC
     AECI               Associated Electric Coop Inc
     AVRN                    Avangrid Renewables LCC
      AVA                                Avista Corp
     BANC Balancing Authority of Northern California
     BPAT            Bonneville Power Administration


## 3. Spatial Join: Assign Each Plant to Its BA

In [7]:
# ── Load power plants from notebook 01 ───────────────────────────────────────
plants_path = PROJECT_ROOT / 'data' / 'processed' / 'power_plants.geojson'
plants = gpd.read_file(plants_path)
print(f'Plants loaded: {len(plants):,} rows, CRS: {plants.crs}')

# Ensure numeric capacity
plants['capacity_mw'] = pd.to_numeric(plants['nameplate-capacity-mw'], errors='coerce')
print(f'Missing capacity: {plants["capacity_mw"].isna().sum():,}')

Plants loaded: 15,000 rows, CRS: EPSG:4326
Missing capacity: 1


In [8]:
# ── Spatial join: point-in-polygon ───────────────────────────────────────────
# Each plant point is assigned the BA polygon it falls within.
# Plants outside all BA polygons (e.g. Alaska, Hawaii, territories) get NaN.
print('Running spatial join (point-in-polygon)...')
plants_ba = gpd.sjoin(
    plants,
    ba[[BA_CODE_COL, BA_NAME_COL, 'geometry']].rename(
        columns={BA_CODE_COL: 'ba_code', BA_NAME_COL: 'ba_name'}
    ),
    how='left',
    predicate='within'
)

matched = plants_ba['ba_code'].notna().sum()
total = len(plants_ba)
print(f'Matched {matched:,} of {total:,} plants to a BA ({100*matched/total:.1f}%)')
print(f'Unmatched (outside BA coverage): {total - matched:,}')

Running spatial join (point-in-polygon)...
Matched 14,931 of 15,034 plants to a BA (99.3%)
Unmatched (outside BA coverage): 103


In [9]:
# ── Save plants with BA codes attached ───────────────────────────────────────
utils.save_processed(plants_ba, 'power_plants_with_ba.geojson')

Saved 15,034 features → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/power_plants_with_ba.geojson


PosixPath('/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/power_plants_with_ba.geojson')

## 4. BA Capacity Summary

In [10]:
# ── Aggregate capacity by BA and fuel type ────────────────────────────────────
# Pivot: rows = BA, columns = fuel type, values = total nameplate MW
capacity_pivot = (
    plants_ba
    .dropna(subset=['ba_code', 'capacity_mw'])
    .groupby(['ba_code', 'ba_name', 'energy-source-desc'])['capacity_mw']
    .sum()
    .reset_index()
    .pivot_table(
        index=['ba_code', 'ba_name'],
        columns='energy-source-desc',
        values='capacity_mw',
        aggfunc='sum',
        fill_value=0
    )
)
capacity_pivot.columns.name = None
capacity_pivot['total_mw'] = capacity_pivot.sum(axis=1)
capacity_pivot = capacity_pivot.sort_values('total_mw', ascending=False)

print(f'BA capacity summary shape: {capacity_pivot.shape}')
capacity_pivot[['total_mw']].head(15)

BA capacity summary shape: (67, 34)


,,total_mw
ba_code,ba_name,
MISO,Midcontinent ISO (Balancing Authority),166857.6
PJM,PJM Interconnection,165925.3
ERCO,ERCOT ISO (Balancing Authority),88375.1
SWPP,Southwest Power Pool (Balancing Authority),83494.5
SOCO,Southern Co Services Inc,61589.7
CISO,California Independent System Operator (Balancing Authority),60242.1
TVA,Tennessee Valley Authority,41394.9
NYIS,New York ISO (Balancing Authority),38170.2
ISNE,New England ISO (Balancing Authority),29376.1


In [11]:
# ── Save capacity summary CSV ─────────────────────────────────────────────────
csv_path = PROJECT_ROOT / 'data' / 'processed' / 'ba_capacity_summary.csv'
capacity_pivot.to_csv(csv_path)
print(f'Saved → {csv_path}')

Saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/ba_capacity_summary.csv


In [12]:
# ── Merge total_mw back onto BA polygons for mapping ─────────────────────────
ba_summary = ba.rename(columns={BA_CODE_COL: 'ba_code', BA_NAME_COL: 'ba_name'}).copy()
# Reset only ba_code + total_mw to avoid duplicate ba_name column after merge
ba_summary = ba_summary.merge(
    capacity_pivot[['total_mw']].reset_index()[['ba_code', 'total_mw']],
    on='ba_code',
    how='left'
)
ba_summary['total_mw'] = ba_summary['total_mw'].fillna(0)

utils.save_processed(ba_summary, 'ba_territories.geojson')

Saved 85 features → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/ba_territories.geojson


PosixPath('/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/ba_territories.geojson')

## 5. Folium Choropleth Map

In [13]:
# ── Build choropleth: BA territories shaded by total installed MW ─────────────
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=ba_summary.__geo_interface__,
    data=ba_summary.set_index('ba_code')['total_mw'],
    columns=['ba_code', 'total_mw'],
    key_on='feature.properties.ba_code',
    fill_color='YlOrRd',
    fill_opacity=0.65,
    line_opacity=0.4,
    legend_name='Total installed capacity (MW)',
    nan_fill_color='#cccccc',
).add_to(m)

# Add BA name tooltips
folium.GeoJson(
    ba_summary.__geo_interface__,
    tooltip=folium.GeoJsonTooltip(
        fields=['ba_code', 'ba_name', 'total_mw'],
        aliases=['BA code', 'BA name', 'Total MW'],
        localize=True
    ),
    style_function=lambda _: {'fillColor': 'transparent', 'color': 'transparent'},
).add_to(m)

map_path = PROJECT_ROOT / 'data' / 'processed' / 'ba_map.html'
m.save(str(map_path))
print(f'Map saved → {map_path}')

Map saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/ba_map.html
